# Tests de modèles

Les tests de modèles complètent les tests unitaires du code.
Les tests pré-entraînement vérifient la cohérence des données avant l'ajustement.
Les tests post-entraînement inspectent le comportement du modèle calibré : invariance, effet directionnel et prototypes.
Ce notebook les écrit pour le pipeline `processing`, puis les applique au modèle de boosting déjà entraîné.
Il n'existe pas de test universel : chaque contrôle dépend du cas d'usage et du coût d'une mauvaise décision.

## Tests pré-entraînement

Ces tests arrêtent l'entraînement lorsque les données ne respectent pas des critères simples.
Les critères utilisés ici portent sur la colonne `purchased` (valeurs 0 ou 1), le type numérique de toutes les colonnes, un effectif minimal et une part minimale de chaque classe.

`encode_features` encode `primary`. `split_dataset` sépare entraînement et test.
Les constantes `BALANCE_THRESHOLD` (0,1) et `MIN_SAMPLES` (5 000) dépendent du contexte. Un autre jeu peut exiger 1 % ou un effectif plus haut.

Créer `src/tests/pipelines/processing/` avec `conftest.py`, `test_nodes.py` et `test_pipeline.py`.

Les fixtures `project_id` et `primary_folder` sont redéfinies dans ce dossier.
Elles ne sont pas importées depuis `loading/conftest.py` : un test ne dépend pas d'un autre dossier de tests.
`scope="module"` construit chaque fixture une fois par module.
`dataset_not_encoded` charge le CSV de test via `load_csv_from_bucket`, déjà couverte par les tests de `loading`.
Les commentaires `TODO` rappellent de remplacer `PROJET_GCP` et `FICHIER_CSV`.

### Tests des nodes

`test_encode_features` lit la clé `features` du dictionnaire retourné par `encode_features`.
`test_split_dataset` vérifie les tailles des deux sous-ensembles.
`train_test_split` arrondit l'effectif de test avec `np.ceil`. L'assertion reprend ce calcul. Le commentaire dans le fichier pointe la ligne correspondante de scikit-learn.

`kedro test` limité au dossier n'exécute que ces tests. L'option `-s` affiche les sorties standard.

### Test du pipeline

Le catalogue de test fournit `primary` et `params:test_ratio` comme `MemoryDataset`.
Dans le projet, `encode_features` renvoie un dictionnaire mappé vers `dataset` et `transform_pipeline`.
Le runner conserve `dataset` en mémoire pour `split_dataset`. Ce jeu n'a pas à figurer dans le catalogue de test.
Les assertions comparent les effectifs de `X_train` et `y_train`, puis de `X_test` et `y_test`.
Le jeu intermédiaire est déjà couvert par les tests de nodes.

Le fragment ci-dessous est le rappel du cours. Les sorties lues par `test_pipeline` sont `X_train`, `y_train`, `X_test` et `y_test`.
Le fichier `pipeline.py` du projet reste celui du pipeline `processing`.

La commande suivante exécute toute la suite : `test_run.py`, `loading` et `processing`.

## Tests post-entraînement

Ces tests s'appliquent au modèle déjà ajusté.
Trois familles sont utilisées ici : des perturbations dont l'effet doit rester faible (invariance), des perturbations dont l'effet est connu à l'avance (tests directionnels), et des prototypes obtenus par k-médoïdes.

Charger le modèle et le jeu de test, puis calculer les valeurs de Shapley de la classe positive.

In [ ]:
import os
import joblib
import pandas as pd
import shap

model = joblib.load(os.path.expanduser("~/data/model.pkl"))
X_test = pd.read_csv(os.path.expanduser("~/data/X_test.csv"))
y_test = pd.read_csv(os.path.expanduser("~/data/y_test.csv")).values.flatten()

explainer = shap.TreeExplainer(model)
X_shap = X_test.copy()
shap_values = explainer.shap_values(X_shap)[1]

### Tests d'invariance

Une petite perturbation ne doit pas faire varier fortement la probabilité lorsque la variable a peu d'effet au voisinage du point.
Sur l'observation `X_test.loc[4720]`, la probabilité de la classe positive est d'environ 92 %.
Une variation de ±1 € sur `price` la modifie d'environ 0,165 % pour cette ligne.

Le calcul suivant mesure, pour chaque observation de prix strictement supérieur à 1 €, l'écart absolu de probabilité entre un prix diminué de 1 € et un prix augmenté de 1 €.
Le filtre sur le prix évite une valeur négative.

In [ ]:
x_unit = X_test.loc[4720, :].copy()

print("Pos : {:2.3f}%".format(model.predict_proba([x_unit])[0, 1] * 100))

In [ ]:
x_unit = X_test.loc[4720, :].copy()
x_unit['price'] += 1
print("Pos : {:2.3f}%".format(model.predict_proba([x_unit])[0, 1] * 100))

x_unit = X_test.loc[4720, :].copy()
x_unit['price'] -= 1
print("Pos : {:2.3f}%".format(model.predict_proba([x_unit])[0, 1] * 100))

In [ ]:
import numpy as np

# On ne sélectionne que les articles dont le prix est > à 1€, sinon on aurait un prix ... négatif !
X_test_price = X_test[X_test['price'] > 1]
X_test_price_plus = X_test_price.copy()
X_test_price_plus['price'] += 1
X_test_price_minus = X_test_price.copy()
X_test_price_minus['price'] -= 1

y_price = pd.DataFrame()
y_price["y"] = model.predict_proba(X_test_price)[:, 1]
y_price["y+"] = model.predict_proba(X_test_price_plus)[:, 1]
y_price["y-"] = model.predict_proba(X_test_price_minus)[:, 1]
y_price["abs_delta"] = np.abs(y_price["y-"] - y_price["y+"])
y_price.sort_values("abs_delta", ascending=False).head(n=10)

Pour une dizaine d'observations, l'écart atteint 20 % à 30 %.
Les prix de ces lignes sont en général de l'ordre de 20 à 50 €.
Le graphique de Shapley sur les 100 écarts les plus forts résume les variables en cause.

In [ ]:
idxs = list(y_price.sort_values("abs_delta", ascending=False).head(n=100).index)
X_test_price.loc[idxs, :]

In [ ]:
shap.summary_plot(shap_values[idxs, :], X_shap.loc[idxs, :], plot_size=0.8)

L'écart-type et la part des observations dont l'écart reste sous 5 % décrivent l'effet global.
Le tracé des 1 000 plus forts écarts, dans l'ordre décroissant, fait apparaître un coude.
Une variation de l'ordre de 10 % est acceptable dans ce cas. Les observations au-dessus de 20 % s'examinent à part.

In [ ]:
print("Écart-type : {:2.2f}%".format(y_price["abs_delta"].std()))
print("Proportion : {:2.2f}%".format(
    y_price[y_price["abs_delta"] < 0.05].shape[0] / y_price.shape[0] * 100
))

In [ ]:
import matplotlib.pyplot as plt

n_obs = 1000

plt.figure(figsize=(16,10))
plt.plot(
    range(n_obs),
    y_price.sort_values("abs_delta", ascending=False).iloc[:n_obs, -1],
    lw=2
)

### Tests directionnels

La perturbation a ici un effet attendu, à lire avec les autres variables de la même observation.
L'observation `X_test.loc[375]` a une durée de 10 secondes et une probabilité d'achat faible.
Le graphe de force attribue cet effet surtout à `duration`.

In [ ]:
x_unit = X_test.loc[375, :]

model.predict_proba([x_unit])

In [ ]:
shap.force_plot(explainer.expected_value[0], shap_values[375, :], x_unit, matplotlib=True)

Ajouter 60 secondes à `duration` augmente la probabilité d'environ 70 %.
Cette combinaison est peu présente dans l'échantillon.
Les autres variables de la ligne et le graphe de dépendance de `duration` montrent une interaction forte avec `num_views_session` lorsque la durée est faible.

In [ ]:
x_unit = X_test.iloc[375, :].copy()
x_unit['duration'] += 60

model.predict_proba([x_unit])

In [ ]:
X_test.iloc[375, :]

In [ ]:
shap.dependence_plot("duration", shap_values, X_shap)

Augmenter en même temps `duration` de 10 et `num_views_session` de 10 laisse la probabilité très faible.
Le contrôle directionnel porte donc sur les sessions courtes et peu vues.

In [ ]:
x_unit = X_test.iloc[375, :].copy()
x_unit['duration'] += 10
x_unit['num_views_session'] += 10

model.predict_proba([x_unit])

L'observation `X_test.loc[4720]` a une probabilité élevée.
Retirer 60 secondes à `duration` a un effet plus faible que sur l'observation précédente.
Le seuil de variation se fixe après lecture de l'observation. Une durée de départ de 10 secondes et un ajout de 60 secondes construisent un point peu représenté.

In [ ]:
x_unit = X_test.loc[4720, :]
 
model.predict_proba([x_unit])

In [ ]:
shap.force_plot(explainer.expected_value[0], shap_values[4720, :], x_unit, matplotlib=True)

In [ ]:
x_unit = X_test.loc[4720, :].copy()
x_unit['duration'] -= 60
    
model.predict_proba([x_unit])

### Prototypes

Un prototype est une observation représentative d'un groupe.
Le k-médoïde choisit, dans chaque groupe, le point dont la distance aux autres points du groupe est minimale.
Dix groupes sont ajustés sur `X_test`. Les centres sont les médoïdes.

In [ ]:
!pip install scikit-learn-extra -q

In [ ]:
from sklearn_extra.cluster import KMedoids

kmed = KMedoids(n_clusters=10)
kmed.fit(X_test)

In [ ]:
X_prototypes = pd.DataFrame(
    data=kmed.cluster_centers_,
    columns=X_test.columns
)
X_prototypes

Sauf le premier centre, les médoïdes concernent la même catégorie et sont prédits en classe positive.
Le dernier centre a deux probabilités proches.
Ces prototypes décrivent une journée de données. L'entraînement du projet utilise un historique de 7 jours.

In [ ]:
model.predict_proba(kmed.cluster_centers_)